In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
REPLAY_JSON_PATH = PROJECT_ROOT / "replay_parser" / "output" / "parsed_match.json"

with open(REPLAY_JSON_PATH, "r") as f:
    replay_data = json.load(f)

print("Top-level keys:", list(replay_data.keys()))
for key, value in replay_data.items():
    print(f"{key}: {len(value)} entries")

Top-level keys: ['players', 'deaths', 'structures_destroyed', 'patron_events', 'mid_boss_kills', 'rift_captures', 'urn_pickups', 'rejuvenator_claims', 'stat_snapshots']
players: 12 entries
deaths: 120 entries
structures_destroyed: 19 entries
patron_events: 1 entries
mid_boss_kills: 3 entries
rift_captures: 6 entries
urn_pickups: 10 entries
rejuvenator_claims: 18 entries
stat_snapshots: 205 entries


In [2]:
# team for players
players_df = pd.DataFrame(replay_data["players"])
controller_to_team = dict(zip(players_df["controller_index"], players_df["team"]))
print(players_df)

# long format DataFrame
rows = []
for snapshot in replay_data["stat_snapshots"]:
    t = snapshot["game_time_s"]
    for p in snapshot["players"]:
        rows.append({
            "game_time_s": t,
            "controller_index": p["controller_index"],
            "team": p["team"],
            "kills": p["kills"],
            "deaths": p["deaths"],
            "assists": p["assists"],
            "hero_damage": p["hero_damage"],
            "objective_damage": p["objective_damage"],
            "gold_net_worth": p["gold_net_worth"],
        })

snapshots_long = pd.DataFrame(rows)
print(snapshots_long.shape)
snapshots_long.head(10)

    controller_index  pawn_index  team            username
0                  2        3429     2             seiryuu
1                  3        3290     2             Numlegs
2                  4        3568     2  Demui on Twitch :D
3                  5        3495     2                Tech
4                  6          77     3               Docha
5                  7          84     3              Specty
6                  8        3475     2            Seikatsu
7                  9          97     3                Anri
8                 10          98     3               Robin
9                 11          94     3     Little Boy 1945
10                12          89     2              Golova
11                13        3534     3          privativev
(2435, 9)


,game_time_s,controller_index,team,kills,deaths,assists,hero_damage,objective_damage,gold_net_worth
0,15.03125,5,2,0,0,0,0,0,600
1,15.03125,6,3,0,0,0,0,0,600
2,15.03125,10,3,0,0,0,0,0,600
3,15.03125,11,3,0,0,0,0,0,600
4,15.03125,13,3,0,0,0,0,0,600
5,30.03125,5,2,0,0,0,0,0,600
6,30.03125,6,3,0,0,0,0,0,600
7,30.03125,10,3,0,0,0,0,0,600
8,30.03125,11,3,0,0,0,0,0,600
9,30.03125,12,2,0,0,0,0,0,600


In [3]:
team_snapshots = (
    snapshots_long
    .groupby(["game_time_s", "team"], as_index=False)
    .agg({
        "kills": "sum",
        "deaths": "sum",
        "assists": "sum",
        "hero_damage": "sum",
        "objective_damage": "sum",
        "gold_net_worth": "sum",
    })
)

print(team_snapshots.shape)
print(team_snapshots.head(20))

(408, 8)
    game_time_s  team  kills  deaths  assists  hero_damage  objective_damage  \
0      15.03125     2      0       0        0            0                 0   
1      15.03125     3      0       0        0            0                 0   
2      30.03125     2      0       0        0            0                 0   
3      30.03125     3      0       0        0            0                 0   
4      45.03125     2      0       0        0            0                 0   
5      45.03125     3      0       0        0            0                 0   
6      60.03125     2      0       0        0          213                 0   
7      60.03125     3      0       0        0          300                 0   
8      75.03125     2      0       0        0          887                 0   
9      75.03125     3      0       0        0          640                 0   
10     90.03125     2      0       0        0         1071                 0   
11     90.03125     3      0   

In [4]:
print(team_snapshots.tail(10))

     game_time_s  team  kills  deaths  assists  hero_damage  objective_damage  \
398  3000.046875     2     55      55      110       292164             21858   
399  3000.046875     3     55      55      106       264389             60243   
400  3015.046875     2     55      55      110       296633             21858   
401  3015.046875     3     55      55      106       267175             64569   
402  3030.046875     2     56      57      111       303760             21858   
403  3030.046875     3     57      56      116       277521             67049   
404  3045.046875     2     56      57      111       303760             21858   
405  3045.046875     3     57      56      116       277521             67049   
406  3060.046875     2     56      57      111       303760             21858   
407  3060.046875     3     57      56      116       277521             67049   

     gold_net_worth  
398          315532  
399          376216  
400          315661  
401          389154 

In [5]:
team_wide = (
    team_snapshots
    .pivot(index="game_time_s", columns="team")
)

team_wide.columns = [
    f"{stat}_team{team}"
    for stat, team in team_wide.columns
]

team_wide = team_wide.reset_index()

print(team_wide.shape)
print(team_wide.head(10))

(204, 13)
   game_time_s  kills_team2  kills_team3  deaths_team2  deaths_team3  \
0     15.03125            0            0             0             0   
1     30.03125            0            0             0             0   
2     45.03125            0            0             0             0   
3     60.03125            0            0             0             0   
4     75.03125            0            0             0             0   
5     90.03125            0            0             0             0   
6    105.03125            0            0             0             0   
7    120.03125            0            0             0             0   
8    135.03125            1            0             0             1   
9    150.03125            1            0             0             1   

   assists_team2  assists_team3  hero_damage_team2  hero_damage_team3  \
0              0              0                  0                  0   
1              0              0                  0 

In [6]:
diff_df = pd.DataFrame({
    "game_time_s": team_wide["game_time_s"],

    "kills_diff": (
        team_wide["kills_team2"] -
        team_wide["kills_team3"]
    ),

    "deaths_diff": (
        team_wide["deaths_team2"] -
        team_wide["deaths_team3"]
    ),

    "assists_diff": (
        team_wide["assists_team2"] -
        team_wide["assists_team3"]
    ),

    "hero_damage_diff": (
        team_wide["hero_damage_team2"] -
        team_wide["hero_damage_team3"]
    ),

    "objective_damage_diff": (
        team_wide["objective_damage_team2"] -
        team_wide["objective_damage_team3"]
    ),

    "net_worth_diff": (
        team_wide["gold_net_worth_team2"] -
        team_wide["gold_net_worth_team3"]
    ),
})

print(diff_df.shape)
print(diff_df.head(10))

(204, 7)
   game_time_s  kills_diff  deaths_diff  assists_diff  hero_damage_diff  \
0     15.03125           0            0             0                 0   
1     30.03125           0            0             0                 0   
2     45.03125           0            0             0                 0   
3     60.03125           0            0             0               -87   
4     75.03125           0            0             0               247   
5     90.03125           0            0             0               378   
6    105.03125           0            0             0               402   
7    120.03125           0            0             0              -326   
8    135.03125           1           -1             1               154   
9    150.03125           1           -1             1              -381   

   objective_damage_diff  net_worth_diff  
0                      0           -1800  
1                      0           -1200  
2                      0            

In [7]:
# replay event lists into DataFrames
deaths_df = pd.DataFrame(replay_data["deaths"])
structures_df = pd.DataFrame(replay_data["structures_destroyed"])
mid_boss_df = pd.DataFrame(replay_data["mid_boss_kills"])
rejuvenator_df = pd.DataFrame(replay_data["rejuvenator_claims"])

print("Deaths:")
print(deaths_df.head())

print("\nStructures:")
print(structures_df.head())

print("\nMid boss:")
print(mid_boss_df)

print("\nRejuvenator claims:")
print(rejuvenator_df.head(20))

Deaths:
   team  player_index            username  game_time_s  death_duration_s  \
0     3            94     Little Boy 1945   128.187500          8.062500   
1     2            89              Golova   151.484375          8.062500   
2     3            94     Little Boy 1945   214.687500          8.046875   
3     2            92  Demui on Twitch :D   323.109375          8.062500   
4     3            94     Little Boy 1945   324.718750          8.062500   

  respawn_full_health  
0               False  
1               False  
2               False  
3               False  
4               False  

Structures:
   structure_type  team  destroyed_time_s
0  Tier2Structure     2        670.609375
1  Tier2Structure     3        784.265625
2  Tier2Structure     3        869.343750
3  Tier1Structure     2        903.500000
4  Tier2Structure     2        934.390625

Mid boss:
   killed_time_s
0    1598.609375
1    2563.921875
2    2954.109375

Rejuvenator claims:
    controller_index  team

In [8]:
team_mapping = {
    2: "Team0",
    3: "Team1"
}

mid_boss_with_team = []

for _, boss in mid_boss_df.iterrows():
    boss_time = boss["killed_time_s"]

    claims_after_boss = rejuvenator_df[
        (rejuvenator_df["game_time_s"] >= boss_time) &
        (rejuvenator_df["game_time_s"] <= boss_time + 30)
    ].sort_values("game_time_s")

    team_claimed = claims_after_boss.iloc[0]["team"] if len(claims_after_boss) > 0 else None

    mid_boss_with_team.append({
        "killed_time_s": boss_time,
        "team_claimed": team_claimed,
        "perspective_team": team_mapping.get(team_claimed) if team_claimed else None,
    })

mid_boss_df = pd.DataFrame(mid_boss_with_team)
print(mid_boss_df)

   killed_time_s  team_claimed perspective_team
0    1598.609375           2.0            Team0
1    2563.921875           3.0            Team1
2    2954.109375           3.0            Team1


In [9]:
FAST_RESPAWN_THRESHOLD_S = 4.0 # some extra time just in case
REJUV_WINDOW_S = 180  # 3 mins

def team_had_active_rejuv(row, rejuv_df, window_s=REJUV_WINDOW_S):
    team_claims = rejuv_df[rejuv_df["team"] == row["team"]]
    active = team_claims[
        (team_claims["game_time_s"] <= row["game_time_s"]) &
        (row["game_time_s"] <= team_claims["game_time_s"] + window_s)
    ]
    return not active.empty

deaths_df["likely_rejuv_used"] = deaths_df["death_duration_s"] < FAST_RESPAWN_THRESHOLD_S

deaths_df["rejuv_buff_active"] = deaths_df.apply(
    lambda r: team_had_active_rejuv(r, rejuvenator_df), axis=1
)

deaths_df["credit_used"] = deaths_df["likely_rejuv_used"] & deaths_df["rejuv_buff_active"]

rejuv_usage_events = deaths_df[deaths_df["credit_used"]][
    ["team", "player_index", "game_time_s", "death_duration_s"]
]
print(rejuv_usage_events)

     team  player_index  game_time_s  death_duration_s
66      2            78  1705.156250          2.984375
68      2            82  1743.828125          3.000000
71      2            92  1761.109375          2.984375
102     3            94  2729.343750          3.000000
103     3            77  2733.015625          2.984375
113     3           100  2967.468750          3.000000


In [10]:
team_wide = team_wide.rename(columns={
    col: col.replace("_team2", "_team0").replace("_team3", "_team1")
    for col in team_wide.columns
})
print(team_wide.columns.tolist())

['game_time_s', 'kills_team0', 'kills_team1', 'deaths_team0', 'deaths_team1', 'assists_team0', 'assists_team1', 'hero_damage_team0', 'hero_damage_team1', 'objective_damage_team0', 'objective_damage_team1', 'gold_net_worth_team0', 'gold_net_worth_team1']


In [11]:
structures_df["destroying_team"] = structures_df["team"].map({2: 3, 3: 2})  # whoever didn't own it destroyed it
structures_df["perspective_team"] = structures_df["destroying_team"].map(team_mapping)

# cumulative count over time per team
structures_df = structures_df.sort_values("destroyed_time_s")
structures_df["team0_cumulative"] = (structures_df["perspective_team"] == "Team0").cumsum()
structures_df["team1_cumulative"] = (structures_df["perspective_team"] == "Team1").cumsum()
print(structures_df[["destroyed_time_s", "team", "destroying_team", "team0_cumulative", "team1_cumulative"]])

    destroyed_time_s  team  destroying_team  team0_cumulative  \
0         670.609375     2                3                 0   
1         784.265625     3                2                 1   
2         869.343750     3                2                 2   
3         903.500000     2                3                 2   
4         934.390625     2                3                 2   
5         980.875000     2                3                 2   
6        1129.593750     3                2                 3   
7        1191.781250     3                2                 4   
8        1633.406250     2                3                 4   
9        1672.765625     3                2                 5   
10       1778.203125     2                3                 5   
11       2174.671875     2                3                 5   
12       2347.703125     2                3                 5   
13       2567.593750     2                3                 5   
14       2572.781250     

In [12]:
mid_boss_df = mid_boss_df.sort_values("killed_time_s")
mid_boss_df["team0_cumulative"] = (mid_boss_df["perspective_team"] == "Team0").cumsum()
mid_boss_df["team1_cumulative"] = (mid_boss_df["perspective_team"] == "Team1").cumsum()
print(mid_boss_df)

   killed_time_s  team_claimed perspective_team  team0_cumulative  \
0    1598.609375           2.0            Team0                 1   
1    2563.921875           3.0            Team1                 1   
2    2954.109375           3.0            Team1                 1   

   team1_cumulative  
0                 0  
1                 1  
2                 2  


In [13]:
team_wide = team_wide.sort_values("game_time_s")
structures_df = structures_df.sort_values("destroyed_time_s")
mid_boss_df = mid_boss_df.sort_values("killed_time_s")

merged = pd.merge_asof(
    team_wide,
    structures_df[["destroyed_time_s", "team0_cumulative", "team1_cumulative"]].rename(
        columns={"team0_cumulative": "own_objectives_cumulative", "team1_cumulative": "enemy_objectives_cumulative"}
    ),
    left_on="game_time_s",
    right_on="destroyed_time_s",
    direction="backward",
)

merged = pd.merge_asof(
    merged,
    mid_boss_df[["killed_time_s", "team0_cumulative", "team1_cumulative"]].rename(
        columns={"team0_cumulative": "own_mid_boss_claims", "team1_cumulative": "enemy_mid_boss_claims"}
    ),
    left_on="game_time_s",
    right_on="killed_time_s",
    direction="backward",
)

# fill NaNs with 0
for col in ["own_objectives_cumulative", "enemy_objectives_cumulative", "own_mid_boss_claims", "enemy_mid_boss_claims"]:
    merged[col] = merged[col].fillna(0).astype("int64")

merged = merged.drop(columns=["destroyed_time_s", "killed_time_s"])
print(merged.shape)
print(merged.head(10))

(204, 17)
   game_time_s  kills_team0  kills_team1  deaths_team0  deaths_team1  \
0     15.03125            0            0             0             0   
1     30.03125            0            0             0             0   
2     45.03125            0            0             0             0   
3     60.03125            0            0             0             0   
4     75.03125            0            0             0             0   
5     90.03125            0            0             0             0   
6    105.03125            0            0             0             0   
7    120.03125            0            0             0             0   
8    135.03125            1            0             0             1   
9    150.03125            1            0             0             1   

   assists_team0  assists_team1  hero_damage_team0  hero_damage_team1  \
0              0              0                  0                  0   
1              0              0                  0 

In [14]:
merged['net_worth_diff'] = merged['gold_net_worth_team0'] - merged['gold_net_worth_team1']
merged['kills_diff'] = merged['kills_team0'] - merged['kills_team1']
merged['deaths_diff'] = merged['deaths_team0'] - merged['deaths_team1']
merged['damage_diff'] = merged['hero_damage_team0'] - merged['hero_damage_team1']
merged['objectives_diff'] = merged['own_objectives_cumulative'] - merged['enemy_objectives_cumulative']
merged['deaths_cumulative_diff'] = merged['deaths_team0'] - merged['deaths_team1']
merged['mid_boss_diff'] = merged['own_mid_boss_claims'] - merged['enemy_mid_boss_claims']

feature_cols = [
    'own_deaths_cumulative', 'enemy_deaths_cumulative',
    'own_objectives_cumulative', 'enemy_objectives_cumulative',
    'own_mid_boss_claims', 'enemy_mid_boss_claims',
    'net_worth_diff', 'kills_diff', 'deaths_diff', 'damage_diff',
    'objectives_diff', 'deaths_cumulative_diff', 'mid_boss_diff'
]

# rename deaths_team0/1 to match training column names for cumulative deaths
merged['own_deaths_cumulative'] = merged['deaths_team0']
merged['enemy_deaths_cumulative'] = merged['deaths_team1']

print(merged[feature_cols].shape)
merged[feature_cols].head(10)

(204, 13)


,own_deaths_cumulative,enemy_deaths_cumulative,own_objectives_cumulative,enemy_objectives_cumulative,own_mid_boss_claims,enemy_mid_boss_claims,net_worth_diff,kills_diff,deaths_diff,damage_diff,objectives_diff,deaths_cumulative_diff,mid_boss_diff
0,0,0,0,0,0,0,-1800,0,0,0,0,0,0
1,0,0,0,0,0,0,-1200,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,-87,0,0,0
4,0,0,0,0,0,0,-426,0,0,247,0,0,0
5,0,0,0,0,0,0,-204,0,0,378,0,0,0
6,0,0,0,0,0,0,-645,0,0,402,0,0,0
7,0,0,0,0,0,0,-257,0,0,-326,0,0,0
8,0,1,0,0,0,0,310,1,-1,154,0,-1,0
9,0,1,0,0,0,0,386,1,-1,-381,0,-1,0


In [15]:
import torch
import torch.nn as nn

class WinProbabilityLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        logits = self.fc(lstm_out)
        return logits.squeeze(-1)

MODEL_PATH = PROJECT_ROOT / "models" / "win_probability_lstm_v1.pt"
checkpoint = torch.load(MODEL_PATH, map_location="cpu")

trained_feature_cols = checkpoint["feature_cols"]
print("Trained feature order:", trained_feature_cols)
print("Your feature order:   ", feature_cols)
assert trained_feature_cols == feature_cols, "Column order mismatch — reorder before proceeding!"

model = WinProbabilityLSTM(
    input_size=checkpoint["input_size"],
    hidden_size=checkpoint["hidden_size"],
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

feature_array = merged[feature_cols].values.astype("float32")
input_tensor = torch.tensor(feature_array).unsqueeze(0)  # [1, seq_len, input_size]
print("Input shape:", input_tensor.shape)

with torch.no_grad():
    logits = model(input_tensor)
    win_probs = torch.sigmoid(logits).squeeze(0).numpy()

merged["win_prob_team0"] = win_probs
print(merged[["game_time_s", "win_prob_team0"]])

Trained feature order: ['own_deaths_cumulative', 'enemy_deaths_cumulative', 'own_objectives_cumulative', 'enemy_objectives_cumulative', 'own_mid_boss_claims', 'enemy_mid_boss_claims', 'net_worth_diff', 'kills_diff', 'deaths_diff', 'damage_diff', 'objectives_diff', 'deaths_cumulative_diff', 'mid_boss_diff']
Your feature order:    ['own_deaths_cumulative', 'enemy_deaths_cumulative', 'own_objectives_cumulative', 'enemy_objectives_cumulative', 'own_mid_boss_claims', 'enemy_mid_boss_claims', 'net_worth_diff', 'kills_diff', 'deaths_diff', 'damage_diff', 'objectives_diff', 'deaths_cumulative_diff', 'mid_boss_diff']
Input shape: torch.Size([1, 204, 13])
     game_time_s  win_prob_team0
0      15.031250        0.188125
1      30.031250        0.177117
2      45.031250        0.352302
3      60.031250        0.416144
4      75.031250        0.401653
..           ...             ...
199  3000.046875        0.326755
200  3015.046875        0.303769
201  3030.046875        0.339255
202  3045.046875

In [16]:
merged["win_prob_delta"] = merged["win_prob_team0"].diff()

# flag meaningful swings — aka when an event happens to de/increase the win prob
SWING_THRESHOLD = 0.05

swings = merged[merged["win_prob_delta"].abs() >= SWING_THRESHOLD][
    ["game_time_s", "win_prob_team0", "win_prob_delta"]
]
print(f"Found {len(swings)} swings of {SWING_THRESHOLD}+ magnitude:")
print(swings)

print("\nDelta distribution (to help pick a good threshold):")
print(merged["win_prob_delta"].describe())

Found 43 swings of 0.05+ magnitude:
     game_time_s  win_prob_team0  win_prob_delta
2      45.031250        0.352302        0.175185
3      60.031250        0.416144        0.063842
6     105.031250        0.327984       -0.079169
7     120.031250        0.245233       -0.082751
8     135.031250        0.520308        0.275075
9     150.031250        0.664794        0.144485
10    165.031250        0.458741       -0.206052
11    180.031250        0.272689       -0.186052
12    195.031250        0.159037       -0.113653
14    225.031250        0.596993        0.432030
15    240.046875        0.651483        0.054490
16    255.046875        0.576222       -0.075261
17    270.046875        0.388541       -0.187681
18    285.046875        0.457748        0.069207
19    300.046875        0.797832        0.340083
30    465.046875        0.756987        0.057520
31    480.046875        0.659271       -0.097716
32    495.046875        0.535890       -0.123381
34    525.046875        0.621469 

In [17]:
# smoothing the noise, esp in early game

merged["win_prob_smoothed"] = merged["win_prob_team0"].rolling(window=3, center=True, min_periods=1).mean()
merged["win_prob_delta_smoothed"] = merged["win_prob_smoothed"].diff()

swings_smoothed = merged[merged["win_prob_delta_smoothed"].abs() >= SWING_THRESHOLD][
    ["game_time_s", "win_prob_smoothed", "win_prob_delta_smoothed"]
]
print(f"Found {len(swings_smoothed)} smoothed swings of {SWING_THRESHOLD}+ magnitude:")
print(swings_smoothed)

Found 31 smoothed swings of 0.05+ magnitude:
     game_time_s  win_prob_smoothed  win_prob_delta_smoothed
1      30.031250           0.239181                 0.056560
2      45.031250           0.315188                 0.076006
3      60.031250           0.390033                 0.074845
6     105.031250           0.326790                -0.052140
8     135.031250           0.476778                 0.112270
9     150.031250           0.547948                 0.071169
10    165.031250           0.465408                -0.082540
11    180.031250           0.296822                -0.168586
12    195.031250           0.198896                -0.097926
13    210.031250           0.306997                 0.108101
14    225.031250           0.471146                 0.164149
15    240.046875           0.608233                 0.137086
16    255.046875           0.538749                -0.069484
17    270.046875           0.474171                -0.064578
18    285.046875           0.548040     

In [18]:
urn_df = pd.DataFrame(replay_data["urn_pickups"])
rift_df = pd.DataFrame(replay_data["rift_captures"])

print(urn_df.head())
print(rift_df.head())

   spawn_time_s  pickup_time_s
0     630.03125     681.796875
1     930.03125     942.671875
2    1230.03125    1251.968750
3    1530.03125    1766.250000
4    1830.03125    1842.828125
   cashin_time_s  spawn_time_s
0     744.093750    724.000000
1    1148.296875   1128.203125
2    1659.015625   1638.921875
3    1984.687500   1964.593750
4    2520.796875   2500.703125


In [19]:
def infer_team_from_net_worth_spike(event_time, window_s=5):
    before = merged[merged["game_time_s"] <= event_time].tail(1)
    after = merged[merged["game_time_s"] >= event_time].head(1)

    if before.empty or after.empty:
        return None

    net_worth_diff_before = before["net_worth_diff"].values[0]
    net_worth_diff_after = after["net_worth_diff"].values[0]
    delta = net_worth_diff_after - net_worth_diff_before

    # positive delta = team0 (team 2) gained relatively -> team0 likely claimed it
    # negative delta = team1 (team 3) gained relatively -> team1 likely claimed it
    if delta > 0:
        return "Team0"
    elif delta < 0:
        return "Team1"
    return None

rift_df["inferred_team"] = rift_df["cashin_time_s"].apply(infer_team_from_net_worth_spike)
urn_df["inferred_team"] = urn_df["pickup_time_s"].apply(infer_team_from_net_worth_spike)

print(rift_df[["cashin_time_s", "inferred_team"]])
print(urn_df[["pickup_time_s", "inferred_team"]])

   cashin_time_s inferred_team
0     744.093750         Team0
1    1148.296875         Team1
2    1659.015625         Team1
3    1984.687500         Team1
4    2520.796875         Team1
5    2986.015625         Team1
   pickup_time_s inferred_team
0     681.796875         Team0
1     942.671875         Team0
2    1251.968750         Team1
3    1766.250000         Team0
4    1842.828125         Team0
5    2333.796875         Team1
6    2349.875000         Team1
7    2366.312500         Team0
8    2568.265625         Team1
9    2905.218750         Team1


In [20]:
pawn_to_username = {
    p["pawn_index"]: p["username"]
    for p in replay_data["players"]
    if p.get("pawn_index") is not None
}
print(pawn_to_username)

{3429: 'seiryuu', 3290: 'Numlegs', 3568: 'Demui on Twitch :D', 3495: 'Tech', 77: 'Docha', 84: 'Specty', 3475: 'Seikatsu', 97: 'Anri', 98: 'Robin', 94: 'Little Boy 1945', 89: 'Golova', 3534: 'privativev'}


In [22]:
print(deaths_df[["player_index", "username"]].drop_duplicates())
print("\nDeaths with missing username:", deaths_df["username"].isna().sum())

    player_index            username
0             94     Little Boy 1945
1             89              Golova
3             92  Demui on Twitch :D
5             97                Anri
6             82             seiryuu
9             85            Seikatsu
13            77               Docha
17           100          privativev
24            84              Specty
31            98               Robin
46            80                Tech
50            78             Numlegs

Deaths with missing username: 0


In [27]:
def find_nearest_events(swing_time, lookback_s=30):
    events = []

    nearby_deaths = deaths_df[
        (deaths_df["game_time_s"] <= swing_time) &
        (deaths_df["game_time_s"] >= swing_time - lookback_s)
    ]
    for _, d in nearby_deaths.iterrows():
        events.append({
            "type": "death",
            "time": d["game_time_s"],
            "team": d["team"],
            "player_index": d["player_index"],
            "username": d["username"],
        })

    nearby_structures = structures_df[
        (structures_df["destroyed_time_s"] <= swing_time) &
        (structures_df["destroyed_time_s"] >= swing_time - lookback_s)
    ]
    for _, s in nearby_structures.iterrows():
        events.append({"type": "structure", "time": s["destroyed_time_s"], "team": s["team"], "structure_type": s["structure_type"]})

    nearby_midboss = mid_boss_df[
        (mid_boss_df["killed_time_s"] <= swing_time) &
        (mid_boss_df["killed_time_s"] >= swing_time - lookback_s)
    ]
    for _, mb in nearby_midboss.iterrows():
        events.append({"type": "mid_boss", "time": mb["killed_time_s"], "team_claimed": mb["team_claimed"]})

    nearby_rejuv = rejuvenator_df[
        (rejuvenator_df["game_time_s"] <= swing_time) &
        (rejuvenator_df["game_time_s"] >= swing_time - lookback_s)
    ]
    for _, r in nearby_rejuv.iterrows():
        events.append({"type": "rejuv_claim", "time": r["game_time_s"], "team": r["team"]})

    nearby_urns = urn_df[
        (urn_df["pickup_time_s"] <= swing_time) &
        (urn_df["pickup_time_s"] >= swing_time - lookback_s)
    ]
    for _, u in nearby_urns.iterrows():
        events.append({"type": "urn_pickup", "time": u["pickup_time_s"], "team": u["inferred_team"]})

    nearby_rifts = rift_df[
        (rift_df["cashin_time_s"] <= swing_time) &
        (rift_df["cashin_time_s"] >= swing_time - lookback_s)
    ]
    for _, r in nearby_rifts.iterrows():
        events.append({"type": "rift_capture", "time": r["cashin_time_s"], "team": r["inferred_team"]})

    return sorted(events, key=lambda e: e["time"], reverse=True)


attributed_swings = []

for _, row in swings_smoothed.iterrows():
    swing_time = row["game_time_s"]
    delta = row["win_prob_delta_smoothed"]
    events = find_nearest_events(swing_time)

    attributed_swings.append({
        "swing_time_s": swing_time,
        "win_prob_after": row["win_prob_smoothed"],
        "delta": delta,
        "direction": "team0_favored" if delta > 0 else "team1_favored",
        "nearest_events": events,
    })

for s in attributed_swings:
    print(f"\n--- Swing at {s['swing_time_s']:.1f}s ({s['direction']}, Δ={s['delta']:+.3f}) ---")
    if not s["nearest_events"]:
        print("  No nearby events found within lookback window")
    for e in s["nearest_events"]:
        print(f"  {e}")


--- Swing at 30.0s (team0_favored, Δ=+0.057) ---
  No nearby events found within lookback window

--- Swing at 45.0s (team0_favored, Δ=+0.076) ---
  No nearby events found within lookback window

--- Swing at 60.0s (team0_favored, Δ=+0.075) ---
  No nearby events found within lookback window

--- Swing at 105.0s (team1_favored, Δ=-0.052) ---
  No nearby events found within lookback window

--- Swing at 135.0s (team0_favored, Δ=+0.112) ---
  {'type': 'death', 'time': 128.1875, 'team': 3, 'player_index': 94, 'username': 'Little Boy 1945'}

--- Swing at 150.0s (team0_favored, Δ=+0.071) ---
  {'type': 'death', 'time': 128.1875, 'team': 3, 'player_index': 94, 'username': 'Little Boy 1945'}

--- Swing at 165.0s (team1_favored, Δ=-0.083) ---
  {'type': 'death', 'time': 151.484375, 'team': 2, 'player_index': 89, 'username': 'Golova'}

--- Swing at 180.0s (team1_favored, Δ=-0.169) ---
  {'type': 'death', 'time': 151.484375, 'team': 2, 'player_index': 89, 'username': 'Golova'}

--- Swing at 195

In [37]:
def get_user_team(username, players_df, team_mapping):
    match = players_df[players_df["username"] == username]
    if match.empty:
        raise ValueError(f"No player found with username '{username}'")
    raw_team = match.iloc[0]["team"]  # this is 2 or 3 (Deadlock's raw team id)
    perspective_team = team_mapping[raw_team]  # "Team0" or "Team1"
    return 0 if perspective_team == "Team0" else 1

# let the user pick their name
USER_USERNAME = "seiryuu" 
USER_TEAM = get_user_team(USER_USERNAME, players_df, team_mapping)
print(f"Viewing report as {USER_USERNAME}, team {USER_TEAM}")

Viewing report as seiryuu, team 0


In [40]:
# early drafttttt

def format_time(seconds):
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m}:{s:02d}"

def event_to_phrase(event, user_team):
    is_user_team = event.get("team") == user_team

    if event["type"] == "death":
        who = event["username"]
        possessive = "your teammate" if is_user_team else "the enemy"
        return f"{possessive} {who} died at {format_time(event['time'])}"

    if event["type"] == "structure":
        side = "your team" if is_user_team else "the enemy"
        verb = "lost" if is_user_team else "destroyed"
        return f"{side} {verb} a {event['structure_type']} at {format_time(event['time'])}"

    if event["type"] == "mid_boss":
        claimer = "your team" if event["team_claimed"] == user_team else "the enemy"
        return f"{claimer} claimed the Mid Boss Rejuvenator at {format_time(event['time'])}"

    if event["type"] == "urn_pickup":
        claimer = "your team" if event["team"] == user_team else "the enemy"
        return f"{claimer} secured the Urn at {format_time(event['time'])}"

    if event["type"] == "rift_capture":
        claimer = "your team" if event["team"] == user_team else "the enemy"
        return f"{claimer} cashed in the Rift at {format_time(event['time'])}"

    return None

def get_win_prob_pct(raw_win_prob_team0, user_team):
    raw = raw_win_prob_team0 if user_team == 0 else (1 - raw_win_prob_team0)
    return raw * 100

GAP_THRESHOLD_S = 300  # bridge silences longer than 5 minutes with a checkpoint line

def generate_narrative(attributed_swings, user_team, merged_df):
    lines = []
    seen_events = set()

    prev_time = merged_df["game_time_s"].min()

    for s in attributed_swings:
        swing_time = s["swing_time_s"]

        # bridge long gaps with a checkpoint describing the quiet stretch
        gap = swing_time - prev_time
        if gap >= GAP_THRESHOLD_S:
            gap_slice = merged_df[
                (merged_df["game_time_s"] >= prev_time) &
                (merged_df["game_time_s"] <= swing_time)
            ]
            probs = gap_slice["win_prob_smoothed"].apply(lambda p: get_win_prob_pct(p, user_team))
            avg_prob = probs.mean()
            state = "led" if avg_prob >= 50 else "trailed"
            lines.append(
                f"From {format_time(prev_time)} to {format_time(swing_time)}, "
                f"your team {state} comfortably (around {avg_prob:.0f}% win chance) with no major swings."
            )

        delta_pct = abs(s["delta"]) * 100
        prob_pct = get_win_prob_pct(s["win_prob_after"], user_team)
        favored = "boosted your win chance" if (s["direction"] == "team0_favored") == (user_team == 0) else "hurt your win chance"

        event_phrase = None
        for e in s["nearest_events"]:
            key = (e["type"], e["time"])
            if key not in seen_events:
                event_phrase = event_to_phrase(e, user_team)
                seen_events.add(key)
                break

        if event_phrase:
            lines.append(f"At {format_time(swing_time)}, {event_phrase}, which {favored} by {delta_pct:.0f}% (now {prob_pct:.0f}% win chance).")
        else:
            lines.append(f"By {format_time(swing_time)}, a natural farming/economy difference {favored} by {delta_pct:.0f}% (now {prob_pct:.0f}% win chance).")

        prev_time = swing_time

    return lines

narrative_lines = generate_narrative(attributed_swings, USER_TEAM, merged)
for line in narrative_lines:
    print(line)

By 0:30, a natural farming/economy difference boosted your win chance by 6% (now 24% win chance).
By 0:45, a natural farming/economy difference boosted your win chance by 8% (now 32% win chance).
By 1:00, a natural farming/economy difference boosted your win chance by 7% (now 39% win chance).
By 1:45, a natural farming/economy difference hurt your win chance by 5% (now 33% win chance).
At 2:15, the enemy Little Boy 1945 died at 2:08, which boosted your win chance by 11% (now 48% win chance).
By 2:30, a natural farming/economy difference boosted your win chance by 7% (now 55% win chance).
At 2:45, the enemy Golova died at 2:31, which hurt your win chance by 8% (now 47% win chance).
By 3:00, a natural farming/economy difference hurt your win chance by 17% (now 30% win chance).
By 3:15, a natural farming/economy difference hurt your win chance by 10% (now 20% win chance).
By 3:30, a natural farming/economy difference boosted your win chance by 11% (now 31% win chance).
At 3:45, the enemy 